# Image Classification with Random Forest and SVM

This notebook builds an image classification system using a public image dataset stored in `images.zip`.  
It covers:

- dataset loading and preprocessing
- Random Forest with `GridSearchCV`
- model evaluation with accuracy, precision, recall, and F1-score
- confusion matrix visualization
- feature importance visualization
- prediction on a new image
- bonus comparison with SVM

**Classes in the dataset:** dalmatian, dollar_bill, pizza, soccer_ball, sunflower


## 1. Import libraries

In [ ]:

import os
import zipfile
import glob
import warnings
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


## 2. Extract the dataset

Upload `images.zip` to Colab first, or place it in the same working directory.


In [ ]:

zip_path = "images.zip"   # change this if your zip file is stored elsewhere
extract_dir = "dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

data_dir = os.path.join(extract_dir, "images")
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

print("Classes:", class_names)


## 3. Load images, resize, normalize, and create labels

To keep the notebook efficient in Colab, each image is converted to **grayscale** and resized to **16 × 16**.  
That gives 256 features per image after flattening.


In [ ]:

image_size = (16, 16)

X = []
y = []
image_paths = []

for label_index, class_name in enumerate(class_names):
    class_folder = os.path.join(data_dir, class_name)
    for file_path in sorted(glob.glob(os.path.join(class_folder, "*.jpg"))):
        image = Image.open(file_path).convert("L").resize(image_size)
        image_array = np.asarray(image, dtype=np.float32) / 255.0  # normalize
        X.append(image_array.flatten())
        y.append(label_index)
        image_paths.append(file_path)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Number of labels:", y.shape)


## 4. Show sample images

In [ ]:

plt.figure(figsize=(10, 4))

for i, file_path in enumerate(image_paths[:10]):
    plt.subplot(2, 5, i + 1)
    image = Image.open(file_path).convert("RGB").resize((64, 64))
    plt.imshow(image)
    plt.title(os.path.basename(os.path.dirname(file_path)), fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 5. Split into training and testing sets

In [ ]:

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X, y, image_paths, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 6. Random Forest with GridSearchCV

In [ ]:

rf_param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)

best_rf = rf_grid.best_estimator_


## 7. Evaluate Random Forest

In [ ]:

rf_predictions = best_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions, average="weighted")
rf_recall = recall_score(y_test, rf_predictions, average="weighted")
rf_f1 = f1_score(y_test, rf_predictions, average="weighted")

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1-score:", rf_f1)

print("\nClassification Report:")
print(classification_report(y_test, rf_predictions, target_names=class_names))


## 8. Random Forest confusion matrix

In [ ]:

rf_cm = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(6, 5))
plt.imshow(rf_cm, interpolation="nearest")
plt.title("Random Forest Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha="right")
plt.yticks(tick_marks, class_names)

for i in range(rf_cm.shape[0]):
    for j in range(rf_cm.shape[1]):
        plt.text(j, i, str(rf_cm[i, j]), ha="center", va="center", fontsize=8)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## 9. Feature importance visualization

In [ ]:

feature_importances = best_rf.feature_importances_

# Bar chart of the top 20 most important pixels
top_n = 20
top_indices = np.argsort(feature_importances)[-top_n:]

plt.figure(figsize=(9, 4))
plt.bar(range(top_n), feature_importances[top_indices])
plt.xticks(range(top_n), top_indices, rotation=45)
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Pixel Index")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

# Heatmap of all pixel importances
plt.figure(figsize=(4, 4))
plt.imshow(feature_importances.reshape(image_size), cmap="hot")
plt.title("Random Forest Pixel Importance Heatmap")
plt.colorbar()
plt.axis("off")
plt.tight_layout()
plt.show()


## 10. Predict the class of a new image

In [ ]:

def preprocess_new_image(image_path, target_size=(16, 16)):
    image = Image.open(image_path).convert("L").resize(target_size)
    image_array = np.asarray(image, dtype=np.float32) / 255.0
    return image_array.flatten().reshape(1, -1)

def predict_new_image(image_path, model, class_names):
    processed = preprocess_new_image(image_path, target_size=image_size)
    prediction_index = model.predict(processed)[0]
    return class_names[prediction_index]

# Example: use one test image
new_image_path = paths_test[0]
predicted_label = predict_new_image(new_image_path, best_rf, class_names)

print("Image path:", new_image_path)
print("Actual class:", os.path.basename(os.path.dirname(new_image_path)))
print("Predicted class:", predicted_label)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(new_image_path))
plt.title(f"Predicted: {predicted_label}")
plt.axis("off")
plt.show()


## 11. Bonus: SVM comparison

In [ ]:

svm_param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale"]
}

svm_grid = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1
)

svm_grid.fit(X_train, y_train)

print("Best SVM Parameters:")
print(svm_grid.best_params_)

best_svm = svm_grid.best_estimator_

svm_predictions = best_svm.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_precision = precision_score(y_test, svm_predictions, average="weighted")
svm_recall = recall_score(y_test, svm_predictions, average="weighted")
svm_f1 = f1_score(y_test, svm_predictions, average="weighted")

print("\nSVM Accuracy:", svm_accuracy)
print("SVM Precision:", svm_precision)
print("SVM Recall:", svm_recall)
print("SVM F1-score:", svm_f1)


## 12. Compare Random Forest and SVM

In [ ]:

metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
rf_scores = [rf_accuracy, rf_precision, rf_recall, rf_f1]
svm_scores = [svm_accuracy, svm_precision, svm_recall, svm_f1]

x = np.arange(len(metric_names))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width/2, rf_scores, width, label="Random Forest")
plt.bar(x + width/2, svm_scores, width, label="SVM")

plt.xticks(x, metric_names)
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Summary

This notebook shows a full image classification workflow using scikit-learn.  
In the tested setup, **Random Forest** performed better than **SVM** on this dataset.

You can now:
- upload this notebook to GitHub
- add a `README.md`
- export the notebook to PDF from Colab
- submit both the notebook and the report
